In [9]:
import numpy as np

corpus = [
    "happy",
    "happier",
    "happiness",
    "help",
    "helper",
    "helping"
]

tokens = []

for word in corpus:
    chars = list(word)
    chars.append("</w>")
    tokens.extend(chars)

print("Corpus")
print(corpus)

print("\nTokens")
print(tokens)

vocabulary = sorted(set(tokens))

print("\nVocabulary")
print(vocabulary)

vocab_size = len(vocabulary)

print("\nVocabulary Size :", vocab_size)

token_to_id = {
    token: idx
    for idx, token in enumerate(vocabulary)
}

id_to_token = {
    idx: token
    for token, idx in token_to_id.items()
}

print("\nToken to ID")
print(token_to_id)


token_ids = [
    token_to_id[token]
    for token in tokens
]

print("\nToken IDs")
print(token_ids)


inputs = token_ids[:-1]

targets = token_ids[1:]

print("\nInput Length :", len(inputs))
print("Target Length :", len(targets))

print("\nFirst 10 Input IDs")
print(inputs[:10])

print("\nFirst 10 Target IDs")
print(targets[:10])


sequence_length = len(inputs)

print("\nSequence Length :", sequence_length)

Corpus
['happy', 'happier', 'happiness', 'help', 'helper', 'helping']

Tokens
['h', 'a', 'p', 'p', 'y', '</w>', 'h', 'a', 'p', 'p', 'i', 'e', 'r', '</w>', 'h', 'a', 'p', 'p', 'i', 'n', 'e', 's', 's', '</w>', 'h', 'e', 'l', 'p', '</w>', 'h', 'e', 'l', 'p', 'e', 'r', '</w>', 'h', 'e', 'l', 'p', 'i', 'n', 'g', '</w>']

Vocabulary
['</w>', 'a', 'e', 'g', 'h', 'i', 'l', 'n', 'p', 'r', 's', 'y']

Vocabulary Size : 12

Token to ID
{'</w>': 0, 'a': 1, 'e': 2, 'g': 3, 'h': 4, 'i': 5, 'l': 6, 'n': 7, 'p': 8, 'r': 9, 's': 10, 'y': 11}

Token IDs
[4, 1, 8, 8, 11, 0, 4, 1, 8, 8, 5, 2, 9, 0, 4, 1, 8, 8, 5, 7, 2, 10, 10, 0, 4, 2, 6, 8, 0, 4, 2, 6, 8, 2, 9, 0, 4, 2, 6, 8, 5, 7, 3, 0]

Input Length : 43
Target Length : 43

First 10 Input IDs
[4, 1, 8, 8, 11, 0, 4, 1, 8, 8]

First 10 Target IDs
[1, 8, 8, 11, 0, 4, 1, 8, 8, 5]

Sequence Length : 43


In [13]:
import numpy as np

embedding_dim = 16

np.random.seed(42)

embedding_matrix = np.random.randn(
    vocab_size,
    embedding_dim
)

embedded_inputs = embedding_matrix[inputs]

print("Embedding Matrix Shape :", embedding_matrix.shape)
print("Embedded Inputs Shape :", embedded_inputs.shape)

sequence_length = len(inputs)

position = np.arange(sequence_length).reshape(-1,1)

dimension = np.arange(embedding_dim).reshape(1,-1)

angle_rates = 1 / np.power(
    10000,
    (2*(dimension//2))/embedding_dim
)

angle_rads = position * angle_rates

positional_encoding = np.zeros(
    (sequence_length, embedding_dim)
)

positional_encoding[:,0::2] = np.sin(
    angle_rads[:,0::2]
)

positional_encoding[:,1::2] = np.cos(
    angle_rads[:,1::2]
)

embedded_inputs = embedded_inputs + positional_encoding

print("\nPositional Encoding Shape :",
      positional_encoding.shape)

print("Embedding + Position Shape :",
      embedded_inputs.shape)

Wq = np.random.randn(
    embedding_dim,
    embedding_dim
)

Wk = np.random.randn(
    embedding_dim,
    embedding_dim
)

Wv = np.random.randn(
    embedding_dim,
    embedding_dim
)

Q = embedded_inputs @ Wq
K = embedded_inputs @ Wk
V = embedded_inputs @ Wv

print("\nQuery Shape :",Q.shape)
print("Key Shape :",K.shape)
print("Value Shape :",V.shape)


scores = (
    Q @ K.T
) / np.sqrt(embedding_dim)

print("\nAttention Score Shape :",
      scores.shape)


mask = np.triu(
    np.ones(scores.shape),
    k=1
)

scores = np.where(
    mask==1,
    -1e9,
    scores
)

print("Causal Mask Applied")


def softmax(x):

    x = x - np.max(
        x,
        axis=-1,
        keepdims=True
    )

    exp = np.exp(x)

    return exp / np.sum(
        exp,
        axis=-1,
        keepdims=True
    )

attention_weights = softmax(scores)

print("\nAttention Weight Shape :",
      attention_weights.shape)

context = attention_weights @ V

print("Context Shape :",
      context.shape)

Embedding Matrix Shape : (12, 16)
Embedded Inputs Shape : (43, 16)

Positional Encoding Shape : (43, 16)
Embedding + Position Shape : (43, 16)

Query Shape : (43, 16)
Key Shape : (43, 16)
Value Shape : (43, 16)

Attention Score Shape : (43, 43)
Causal Mask Applied

Attention Weight Shape : (43, 43)
Context Shape : (43, 16)


In [14]:
import numpy as np


Wo = np.random.randn(
    embedding_dim,
    vocab_size
)

bias = np.zeros(vocab_size)

logits = context @ Wo + bias

print("Logits Shape :", logits.shape)

def softmax(x):

    x = x - np.max(
        x,
        axis=1,
        keepdims=True
    )

    exp = np.exp(x)

    return exp / np.sum(
        exp,
        axis=1,
        keepdims=True
    )

probabilities = softmax(logits)

print("Probability Shape :", probabilities.shape)


target_one_hot = np.zeros_like(probabilities)

target_one_hot[
    np.arange(len(targets)),
    targets
] = 1

loss = -np.mean(
    np.sum(
        target_one_hot *
        np.log(probabilities + 1e-9),
        axis=1
    )
)

print("\nCross Entropy Loss :", loss)


learning_rate = 0.01

d_logits = (
    probabilities -
    target_one_hot
) / len(targets)

dWo = context.T @ d_logits

dbias = np.sum(
    d_logits,
    axis=0
)

Wo -= learning_rate * dWo
bias -= learning_rate * dbias

print("Backpropagation Completed")
print("Weights Updated")


predicted_ids = np.argmax(
    probabilities,
    axis=1
)

predicted_tokens = [
    id_to_token[idx]
    for idx in predicted_ids
]

print("\nGenerated Tokens:")
print(predicted_tokens)

print("\nGenerated Text:")
generated_text = ""

for token in predicted_tokens:

    if token == "</w>":
        generated_text += " "
    else:
        generated_text += token

print(generated_text)

print("\n========== PROGRAM COMPLETED ==========")

Logits Shape : (43, 12)
Probability Shape : (43, 12)

Cross Entropy Loss : 15.107330726020873
Backpropagation Completed
Weights Updated

Generated Tokens:
['n', 'n', 'n', 'n', 'n', 'g', 'g', 'n', 'g', 'g', 'n', '</w>', 'n', 'n', 'g', 'n', 'g', 'g', 'g', 'n', 'g', 'n', 'g', 'n', 'g', 'g', 'n', 'n', 'n', 'n', 'g', 'n', 'n', 'n', 'n', 'g', 'g', 'g', 'n', 'n', 'n', 'i', 'n']

Generated Text:
nnnnnggnggn nngngggngngnggnnnngnnnngggnnnin

========== PROGRAM COMPLETED ==========
